# v10 — Oversample todas las clases + FocalLossAlpha

**Mejora sobre v9:** en v9 solo se triplicó mel. Las clases df (110) y vasc (136)
seguían siendo muy pocas → el modelo las confundía con mel.

**Estrategia v10:**
- Oversample de TODAS las clases minoritarias hasta ~1500 imágenes
- FocalLossAlpha balanceado (mismo que v9, ajustado a nuevas frecuencias)
- Augmentación específica por clase (más agresiva para las más raras)
- Mismo esquema de 3 fases con épocas generosas

**Meta:** F1-macro > 0.70 (v9) manteniendo F1-mel > 0.40


## 1. Entorno y datos

In [1]:
import os, gc
import numpy as np
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cv2
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)
import seaborn as sns

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
N_CLASSES   = len(CLASS_NAMES)
MEL_IDX     = CLASS_NAMES.index('mel')
NV_IDX      = CLASS_NAMES.index('nv')
IMG_SIZE    = 224
BATCH_SIZE  = 32
TARGET_N    = 1500  # todas las clases minoritarias llegan a este número

NUMPY_DIR  = '/kaggle/input/datasets/michellercarranza/ham10000-numpy'
OUTPUT_DIR = '/kaggle/working/v10'
os.makedirs(OUTPUT_DIR, exist_ok=True)


TensorFlow: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
print('Cargando datos...')
X_train = np.load(os.path.join(NUMPY_DIR, 'train_X.npy'))
y_train = np.load(os.path.join(NUMPY_DIR, 'train_y.npy'))
X_val   = np.load(os.path.join(NUMPY_DIR, 'validation_X.npy'))
y_val   = np.load(os.path.join(NUMPY_DIR, 'validation_y.npy'))
X_test  = np.load(os.path.join(NUMPY_DIR, 'test_X.npy'))
y_test  = np.load(os.path.join(NUMPY_DIR, 'test_y.npy'))

y_train_int = y_train
y_val_int   = y_val
y_test_int  = y_test

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print('\nDistribución train original:')
for i, name in enumerate(CLASS_NAMES):
    n = np.sum(y_train_int == i)
    print(f'  {name:6s}: {n:5d}')


Cargando datos...
Train: (9577, 224, 224, 3)  Val: (2492, 224, 224, 3)  Test: (1285, 224, 224, 3)

Distribución train original:
  akiec :   315
  bcc   :   487
  bkl   :  1048
  df    :   110
  mel   :  1076
  nv    :  6405
  vasc  :   136


## 2. Hair Removal

In [3]:
def remove_hair(img_uint8, kernel_size=17, threshold=10):
    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, mask = cv2.threshold(blackhat, threshold, 255, cv2.THRESH_BINARY)
    kernel_dilate = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.dilate(mask, kernel_dilate, iterations=1)
    img_bgr = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR)
    inpainted_bgr = cv2.inpaint(img_bgr, mask, inpaintRadius=3, flags=cv2.INPAINT_TELEA)
    return cv2.cvtColor(inpainted_bgr, cv2.COLOR_BGR2RGB)

def preprocess_batch(X_float32):
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    result = np.empty_like(X_float32)
    for i in range(len(X_float32)):
        img_uint8 = np.clip((X_float32[i] * std + mean) * 255, 0, 255).astype(np.uint8)
        img_clean = remove_hair(img_uint8)
        result[i] = (img_clean.astype(np.float32) / 255.0 - mean) / std
    return result

print('Aplicando hair removal...')
CHUNK = 500
X_train_clean = np.empty_like(X_train)
for i in range(0, len(X_train), CHUNK):
    end = min(i + CHUNK, len(X_train))
    X_train_clean[i:end] = preprocess_batch(X_train[i:end])
    if (i // CHUNK + 1) % 5 == 0:
        print(f'  train {end}/{len(X_train)}')
X_val_clean  = preprocess_batch(X_val)
X_test_clean = preprocess_batch(X_test)
del X_train, X_val, X_test
gc.collect()
print('Hair removal completado.')


Aplicando hair removal...
  train 2500/9577
  train 5000/9577
  train 7500/9577
  train 9577/9577
Hair removal completado.


## 3. Oversample de TODAS las clases a TARGET_N=1500

Cada clase con menos de 1500 imágenes se repite cíclicamente hasta llegar a 1500.
nv (6405) no se toca — ya es la clase mayoritaria.


In [4]:
X_parts = [X_train_clean]
y_parts = [y_train_int]

print(f'Oversample de clases con < {TARGET_N} imágenes:')
for i, name in enumerate(CLASS_NAMES):
    idx = np.where(y_train_int == i)[0]
    n   = len(idx)
    if n >= TARGET_N:
        print(f'  {name:6s}: {n:5d} — sin cambio')
        continue
    need  = TARGET_N - n
    # Repetir cíclicamente las imágenes existentes
    reps  = int(np.ceil(need / n))
    extra_idx = np.tile(idx, reps)[:need]
    X_parts.append(X_train_clean[extra_idx])
    y_parts.append(np.full(need, i, dtype=y_train_int.dtype))
    print(f'  {name:6s}: {n:5d} → {TARGET_N} (+{need})')

X_train_aug = np.concatenate(X_parts, axis=0)
y_train_aug = np.concatenate(y_parts, axis=0)
del X_parts, y_parts
gc.collect()

# Mezclar
perm = np.random.permutation(len(X_train_aug))
X_train_aug = X_train_aug[perm]
y_train_aug = y_train_aug[perm]
del perm; gc.collect()

print(f'\nTrain aumentado: {X_train_aug.shape}')
print(f'RAM estimada arrays: {X_train_aug.nbytes / 1e9:.1f} GB')
print('\nDistribución final:')
for i, name in enumerate(CLASS_NAMES):
    n = np.sum(y_train_aug == i)
    print(f'  {name:6s}: {n:5d} ({n/len(y_train_aug)*100:.1f}%)')


Oversample de clases con < 1500 imágenes:
  akiec :   315 → 1500 (+1185)
  bcc   :   487 → 1500 (+1013)
  bkl   :  1048 → 1500 (+452)
  df    :   110 → 1500 (+1390)
  mel   :  1076 → 1500 (+424)
  nv    :  6405 — sin cambio
  vasc  :   136 → 1500 (+1364)

Train aumentado: (15405, 224, 224, 3)
RAM estimada arrays: 9.3 GB

Distribución final:
  akiec :  1500 (9.7%)
  bcc   :  1500 (9.7%)
  bkl   :  1500 (9.7%)
  df    :  1500 (9.7%)
  mel   :  1500 (9.7%)
  nv    :  6405 (41.6%)
  vasc  :  1500 (9.7%)


## 4. Generador con augmentación diferenciada

- **Clases muy raras** (df, vasc, akiec): rotación 360°, zoom, color jitter, flips
- **mel**: rotación 360°, color jitter (igual que v9)
- **Resto**: rotación 90°, flips básicos


In [5]:
# Clases con augmentación agresiva (las más raras)
AGGRESSIVE_CLASSES = {
    CLASS_NAMES.index('df'),
    CLASS_NAMES.index('vasc'),
    CLASS_NAMES.index('akiec'),
    CLASS_NAMES.index('mel'),
}

def augment_image(img, class_idx):
    img = img.copy()
    mean_v = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std_v  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    aggressive = class_idx in AGGRESSIVE_CLASSES

    # Flips
    if np.random.rand() > 0.5: img = np.fliplr(img)
    if np.random.rand() > 0.5: img = np.flipud(img)

    # Rotación
    max_angle = 360 if aggressive else 90
    angle = np.random.uniform(-max_angle, max_angle)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    img_uint8 = np.clip((img * std_v + mean_v) * 255, 0, 255).astype(np.uint8)
    img_uint8 = cv2.warpAffine(img_uint8, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    img = (img_uint8.astype(np.float32) / 255.0 - mean_v) / std_v

    # Color jitter para clases agresivas
    if aggressive:
        alpha = np.random.uniform(0.85, 1.15)
        beta  = np.random.uniform(-0.05, 0.05)
        img   = img * alpha + beta

    return img.astype(np.float32)


class AugmentedGenerator(tf.keras.utils.Sequence):
    def __init__(self, X, y, batch_size, augment=True, shuffle=True):
        self.X, self.y = X, y
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(X))
        if shuffle: np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_batch = np.empty((len(batch_idx), IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        y_batch = self.y[batch_idx]
        for i, j in enumerate(batch_idx):
            img = self.X[j]
            if self.augment:
                img = augment_image(img, self.y[j])
            X_batch[i] = img
        return X_batch, tf.keras.utils.to_categorical(y_batch, N_CLASSES)

    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)


train_gen = AugmentedGenerator(X_train_aug, y_train_aug, BATCH_SIZE)
val_gen   = AugmentedGenerator(X_val_clean, y_val_int, BATCH_SIZE, augment=False, shuffle=False)
print(f'Generadores listos. Train batches: {len(train_gen)}')


Generadores listos. Train batches: 482


## 5. FocalLossAlpha — recalculado con nuevas frecuencias

Alpha proporcional a 1/frecuencia post-oversample, con mel amplificado ×3 y nv reducido ×5.


In [6]:
class FocalLossAlpha(tf.keras.losses.Loss):
    def __init__(self, alpha=None, gamma=2.0, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = tf.constant(
            alpha if alpha is not None else [1/N_CLASSES]*N_CLASSES,
            dtype=tf.float32)

    def call(self, y_true, y_pred):
        y_pred   = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        alpha_t  = tf.reduce_sum(y_true * self.alpha, axis=-1, keepdims=True)
        pt       = tf.reduce_sum(y_true * y_pred,    axis=-1, keepdims=True)
        focal_w  = alpha_t * tf.pow(1.0 - pt, self.gamma)
        ce       = -y_true * tf.math.log(y_pred)
        return tf.reduce_mean(focal_w * tf.reduce_sum(ce, axis=-1, keepdims=True))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'alpha': self.alpha.numpy().tolist(), 'gamma': self.gamma})
        return cfg


# Recalcular alpha con las frecuencias post-oversample
counts_aug = np.array([np.sum(y_train_aug == i) for i in range(N_CLASSES)], dtype=np.float32)
inv_freq   = 1.0 / counts_aug
alpha_v    = inv_freq / inv_freq.sum()

# mel ×3, nv /5
alpha_v[MEL_IDX] *= 3.0
alpha_v[NV_IDX]  /= 5.0
alpha_v = alpha_v / alpha_v.sum()  # renormalizar

print('Alpha por clase (v10):')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:6s}: {alpha_v[i]:.4f}')
print(f'  Suma: {alpha_v.sum():.4f}')
print(f'  Alpha mínimo: {alpha_v.min():.4f} ({"OK" if alpha_v.min() >= 0.01 else "ADVERTENCIA"})')

focal_loss = FocalLossAlpha(alpha=alpha_v.tolist(), gamma=2.0)
print('FocalLossAlpha v10 lista.')


Alpha por clase (v10):
  akiec : 0.1243
  bcc   : 0.1243
  bkl   : 0.1243
  df    : 0.1243
  mel   : 0.3728
  nv    : 0.0058
  vasc  : 0.1243
  Suma: 1.0000
  Alpha mínimo: 0.0058 (ADVERTENCIA)
FocalLossAlpha v10 lista.


## 6. Modelo + Class weights

In [7]:
def build_model():
    base = tf.keras.applications.DenseNet201(
        include_top=False, weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False
    inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base(inputs, training=False)
    x       = tf.keras.layers.GlobalAveragePooling2D()(x)
    x       = tf.keras.layers.Dense(256, activation='relu',
                                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x       = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(N_CLASSES, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs), base

model, base_model = build_model()
print(f'Parámetros: {model.count_params():,}')


74836368/74836368 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Parámetros: 18,815,559


In [8]:
# Class weights: proporcional al desbalance post-oversample
# (ya con oversample el desbalance se redujo mucho, así que pesos más moderados)
counts_aug = np.array([np.sum(y_train_aug == i) for i in range(N_CLASSES)], dtype=np.float32)
base_w = counts_aug.max() / counts_aug

class_weights = {}
for i, name in enumerate(CLASS_NAMES):
    if i == NV_IDX:
        w = float(base_w[i]) / 3.0  # reducir nv aún más
    else:
        w = float(base_w[i])
    class_weights[i] = w
    print(f'  {name:6s}: weight = {w:.3f}')


  akiec : weight = 4.270
  bcc   : weight = 4.270
  bkl   : weight = 4.270
  df    : weight = 4.270
  mel   : weight = 4.270
  nv    : weight = 0.333
  vasc  : weight = 4.270


## 7. Entrenamiento (3 fases)

In [9]:
# ── Fase 1: cabeza, 15 épocas, CrossEntropy ──
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy']
)
cb1 = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'densenet201_v10_phase1_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]
print('Fase 1 (15 épocas)...')
h1 = model.fit(train_gen, validation_data=val_gen, epochs=15,
               callbacks=cb1, class_weight=class_weights, verbose=1)
print('Fase 1 completada.')


Fase 1 (15 épocas)...


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
482/482 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step - accuracy: 0.3187 - loss: 3.9840
Epoch 1: val_accuracy improved from None to 0.44222, saving model to /kaggle/working/v10/densenet201_v10_phase1_best.keras

Epoch 1: finished saving model to /kaggle/working/v10/densenet201_v10_phase1_best.keras
482/482 ━━━━━━━━━━━━━━━━━━━━ 193s 304ms/step - accuracy: 0.4075 - loss: 3.4808 - val_accuracy: 0.4422 - val_loss: 1.6247 - learning_rate: 0.0010
Epoch 2/15
482/482 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step - accuracy: 0.5225 - loss: 2.9012
Epoch 2: val_accuracy improved from 0.44222 to 0.54575, saving model to /kaggle/working/v10/densenet201_v10_phase1_best.keras

Epoch 2: finished saving model to /kaggle/working/v10/densenet201_v10_phase1_best.keras
482/482 ━━━━━━━━━━━━━━━━━━━━ 130s 269ms/step - accuracy: 0.5224 - loss: 2.8475 - val_accuracy: 0.5457 - val_loss: 1.3240 - learning_rate: 0.0010
Epoch 3/15
482/482 ━━━━━━━━━━━━━━━━━━━━ 0s 227ms/step - accuracy: 0.5598 - loss: 2.6310
Epoch 3: val_accura

In [10]:
# ── Fase 2a: descongelar 25 capas + FocalLossAlpha ──
base_model.trainable = True
for layer in base_model.layers[:-25]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(2e-4),
    loss=focal_loss,
    metrics=['accuracy']
)
cb2a = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'densenet201_v10_phase2a_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]
print('Fase 2a (25 épocas)...')
h2a = model.fit(train_gen, validation_data=val_gen, epochs=25,
                callbacks=cb2a, class_weight=class_weights, verbose=1)
print('Fase 2a completada.')


Fase 2a (25 épocas)...
Epoch 1/25
482/482 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step - accuracy: 0.5496 - loss: 0.1832
Epoch 1: val_accuracy improved from None to 0.44904, saving model to /kaggle/working/v10/densenet201_v10_phase2a_best.keras

Epoch 1: finished saving model to /kaggle/working/v10/densenet201_v10_phase2a_best.keras
482/482 ━━━━━━━━━━━━━━━━━━━━ 207s 326ms/step - accuracy: 0.5524 - loss: 0.1571 - val_accuracy: 0.4490 - val_loss: 0.0527 - learning_rate: 2.0000e-04
Epoch 2/25
482/482 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step - accuracy: 0.5594 - loss: 0.1296
Epoch 2: val_accuracy improved from 0.44904 to 0.50040, saving model to /kaggle/working/v10/densenet201_v10_phase2a_best.keras

Epoch 2: finished saving model to /kaggle/working/v10/densenet201_v10_phase2a_best.keras
482/482 ━━━━━━━━━━━━━━━━━━━━ 97s 200ms/step - accuracy: 0.5675 - loss: 0.1262 - val_accuracy: 0.5004 - val_loss: 0.0484 - learning_rate: 2.0000e-04
Epoch 3/25
482/482 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step - accuracy: 0.5852 

In [11]:
# ── Fase 2b: descongelar 100 capas ──
for layer in base_model.layers[:-100]:
    layer.trainable = False
for layer in base_model.layers[-100:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-5),
    loss=focal_loss,
    metrics=['accuracy']
)
cb2b = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'densenet201_v10_phase2b_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4, min_lr=1e-7, verbose=1)
]
print('Fase 2b (35 épocas)...')
h2b = model.fit(train_gen, validation_data=val_gen, epochs=35,
                callbacks=cb2b, class_weight=class_weights, verbose=1)
print('Fase 2b completada.')


Fase 2b (35 épocas)...
Epoch 1/35
482/482 ━━━━━━━━━━━━━━━━━━━━ 0s 271ms/step - accuracy: 0.7663 - loss: 0.0347
Epoch 1: val_accuracy improved from None to 0.63443, saving model to /kaggle/working/v10/densenet201_v10_phase2b_best.keras

Epoch 1: finished saving model to /kaggle/working/v10/densenet201_v10_phase2b_best.keras
482/482 ━━━━━━━━━━━━━━━━━━━━ 257s 377ms/step - accuracy: 0.7620 - loss: 0.0342 - val_accuracy: 0.6344 - val_loss: 0.0268 - learning_rate: 5.0000e-05
Epoch 2/35
482/482 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step - accuracy: 0.7503 - loss: 0.0318
Epoch 2: val_accuracy improved from 0.63443 to 0.63523, saving model to /kaggle/working/v10/densenet201_v10_phase2b_best.keras

Epoch 2: finished saving model to /kaggle/working/v10/densenet201_v10_phase2b_best.keras
482/482 ━━━━━━━━━━━━━━━━━━━━ 97s 201ms/step - accuracy: 0.7612 - loss: 0.0314 - val_accuracy: 0.6352 - val_loss: 0.0259 - learning_rate: 5.0000e-05
Epoch 3/35
482/482 ━━━━━━━━━━━━━━━━━━━━ 0s 236ms/step - accuracy: 0.7663 

## 8. Curvas de entrenamiento

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
all_acc      = h1.history['accuracy']     + h2a.history['accuracy']     + h2b.history['accuracy']
all_val_acc  = h1.history['val_accuracy'] + h2a.history['val_accuracy'] + h2b.history['val_accuracy']
all_loss     = h1.history['loss']         + h2a.history['loss']         + h2b.history['loss']
all_val_loss = h1.history['val_loss']     + h2a.history['val_loss']     + h2b.history['val_loss']
ep = range(1, len(all_acc) + 1)
p1_end  = len(h1.history['accuracy'])
p2a_end = p1_end + len(h2a.history['accuracy'])
for ax, tv, vv, ylabel in [
    (axes[0], all_acc,  all_val_acc,  'Accuracy'),
    (axes[1], all_loss, all_val_loss, 'Loss'),
]:
    ax.plot(ep, tv, label='Train', color='#1a6e35')
    ax.plot(ep, vv, label='Val',   color='#c85a08')
    ax.axvline(p1_end,  color='gray', linestyle=':', alpha=0.7, label='Fase 2a')
    ax.axvline(p2a_end, color='navy', linestyle=':', alpha=0.7, label='Fase 2b')
    ax.set_xlabel('Época'); ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} por época'); ax.legend()
plt.suptitle('v10 — Oversample todas las clases', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves_v10.png'), dpi=130, bbox_inches='tight')
plt.show()
print('Guardada: training_curves_v10.png')


Guardada: training_curves_v10.png


## 9. Búsqueda de umbral óptimo

In [13]:
val_probs = model.predict(X_val_clean, batch_size=64, verbose=1)
thresholds      = np.arange(0.05, 0.60, 0.01)
f1_mel_scores   = []
f1_macro_scores = []
for t in thresholds:
    preds = np.argmax(val_probs, axis=1).copy()
    preds[val_probs[:, MEL_IDX] > t] = MEL_IDX
    f1_mel_scores.append(f1_score(y_val_int==MEL_IDX, preds==MEL_IDX, zero_division=0))
    f1_macro_scores.append(f1_score(y_val_int, preds, average='macro', zero_division=0))
best_t_macro = thresholds[np.argmax(f1_macro_scores)]
best_t_mel   = thresholds[np.argmax(f1_mel_scores)]
print(f'Umbral óptimo F1-macro: t={best_t_macro:.2f}  ({max(f1_macro_scores):.4f})')
print(f'Umbral óptimo F1-mel:   t={best_t_mel:.2f}  ({max(f1_mel_scores):.4f})')
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, f1_macro_scores, label='F1-macro', color='#1a6e35', linewidth=2)
ax.plot(thresholds, f1_mel_scores,   label='F1-mel',   color='#c85a08', linewidth=2)
ax.axvline(best_t_macro, color='#1a6e35', linestyle='--', alpha=0.7,
           label=f'Mejor macro t={best_t_macro:.2f}')
ax.set_xlabel('Umbral para mel'); ax.set_ylabel('F1 Score')
ax.set_title('Búsqueda de umbral óptimo (val set) — v10'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'threshold_search_v10.png'), dpi=150)
plt.show()


39/39 ━━━━━━━━━━━━━━━━━━━━ 63s 958ms/step
Umbral óptimo F1-macro: t=0.50  (0.7867)
Umbral óptimo F1-mel:   t=0.50  (0.5347)


## 10. Evaluación en test set

In [14]:
test_probs = model.predict(X_test_clean, batch_size=64, verbose=1)

# B: argmax directo
y_pred_B = np.argmax(test_probs, axis=1)
acc_B    = accuracy_score(y_test_int, y_pred_B)
f1_B     = f1_score(y_test_int, y_pred_B, average='macro', zero_division=0)
f1_mel_B = f1_score((y_test_int==MEL_IDX), (y_pred_B==MEL_IDX), zero_division=0)
print('=== Estrategia B — argmax ===')
print(classification_report(y_test_int, y_pred_B, target_names=CLASS_NAMES, zero_division=0))
print(f'Accuracy={acc_B:.4f}  F1-macro={f1_B:.4f}  F1-mel={f1_mel_B:.4f}')
print()

# A+B: argmax + umbral
THRESHOLD_MEL = best_t_macro
y_pred_AB = np.argmax(test_probs, axis=1).copy()
y_pred_AB[test_probs[:, MEL_IDX] > THRESHOLD_MEL] = MEL_IDX
acc_AB    = accuracy_score(y_test_int, y_pred_AB)
f1_AB     = f1_score(y_test_int, y_pred_AB, average='macro', zero_division=0)
f1_mel_AB = f1_score((y_test_int==MEL_IDX), (y_pred_AB==MEL_IDX), zero_division=0)
print(f'=== Estrategia A+B — umbral {THRESHOLD_MEL:.2f} ===')
print(classification_report(y_test_int, y_pred_AB, target_names=CLASS_NAMES, zero_division=0))
print(f'Accuracy={acc_AB:.4f}  F1-macro={f1_AB:.4f}  F1-mel={f1_mel_AB:.4f}')


21/21 ━━━━━━━━━━━━━━━━━━━━ 28s 1s/step
=== Estrategia B — argmax ===
              precision    recall  f1-score   support

       akiec       0.94      0.79      0.86        42
         bcc       0.81      0.85      0.83        67
         bkl       0.52      0.92      0.66       142
          df       0.92      0.86      0.89        14
         mel       0.37      0.95      0.53       144
          nv       0.99      0.61      0.76       858
        vasc       0.95      1.00      0.97        18

    accuracy                           0.71      1285
   macro avg       0.79      0.85      0.79      1285
weighted avg       0.86      0.71      0.73      1285

Accuracy=0.7089  F1-macro=0.7862  F1-mel=0.5320

=== Estrategia A+B — umbral 0.50 ===
              precision    recall  f1-score   support

       akiec       0.94      0.79      0.86        42
         bcc       0.81      0.85      0.83        67
         bkl       0.52      0.92      0.66       142
          df       0.92      0.

## 11. Matriz de confusión

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, y_pred, title in [
    (axes[0], y_pred_B,
     f'B — argmax\nAcc={acc_B:.3f} F1={f1_B:.3f} F1-mel={f1_mel_B:.3f}'),
    (axes[1], y_pred_AB,
     f'A+B — umbral {THRESHOLD_MEL:.2f}\nAcc={acc_AB:.3f} F1={f1_AB:.3f} F1-mel={f1_mel_AB:.3f}'),
]:
    cm = confusion_matrix(y_test_int, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='OrRd',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.5, ax=ax)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel('Clase real'); ax.set_xlabel('Clase predicha')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrices_v10.png'), dpi=130, bbox_inches='tight')
plt.show()
print('Guardada: confusion_matrices_v10.png')


Guardada: confusion_matrices_v10.png


## 12. Resumen comparativo

In [16]:
print('=' * 68)
print('COMPARATIVA DE VERSIONES')
print('=' * 68)
print(f'{"Versión":28s} {"Accuracy":>10} {"F1-macro":>10} {"F1-mel":>10}')
print('-' * 63)
print(f'{"v5 (baseline)":28s} {0.6879:>10.4f} {0.4362:>10.4f} {0.0000:>10.4f}')
print(f'{"v8 (alpha extremo)":28s} {0.6864:>10.4f} {0.1933:>10.4f} {0.4797:>10.4f}')
print(f'{"v9 (alpha balanceado)":28s} {0.6010:>10.4f} {0.7020:>10.4f} {0.4370:>10.4f}')
print(f'{"v10-B (oversample todas)":28s} {acc_B:>10.4f} {f1_B:>10.4f} {f1_mel_B:>10.4f}')
print(f'{"v10-AB (+ umbral)":28s} {acc_AB:>10.4f} {f1_AB:>10.4f} {f1_mel_AB:>10.4f}')
print('=' * 68)

best_f1  = max(f1_B, f1_AB)
best_mel = max(f1_mel_B, f1_mel_AB)
print(f'\nMeta F1-macro > 0.70 (v9): {"✓ SUPERADA" if best_f1 > 0.70 else "✗ No superada"} ({best_f1:.4f})')
print(f'Meta F1-mel   > 0.40:      {"✓ SUPERADA" if best_mel > 0.40 else "✗ No superada"} ({best_mel:.4f})')

print('\nArchivos generados:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    print(f'  {f:52s} {os.path.getsize(fp)/1e6:.1f} MB')


COMPARATIVA DE VERSIONES
Versión                        Accuracy   F1-macro     F1-mel
---------------------------------------------------------------
v5 (baseline)                    0.6879     0.4362     0.0000
v8 (alpha extremo)               0.6864     0.1933     0.4797
v9 (alpha balanceado)            0.6010     0.7020     0.4370
v10-B (oversample todas)         0.7089     0.7862     0.5320
v10-AB (+ umbral)                0.7089     0.7862     0.5320

Meta F1-macro > 0.70 (v9): ✓ SUPERADA (0.7862)
Meta F1-mel   > 0.40:      ✓ SUPERADA (0.5320)

Archivos generados:
  confusion_matrices_v10.png                           0.1 MB
  densenet201_v10_phase1_best.keras                    81.7 MB
  densenet201_v10_phase2a_best.keras                   88.7 MB
  densenet201_v10_phase2b_best.keras                   110.4 MB
  threshold_search_v10.png                             0.1 MB
  training_curves_v10.png                              0.1 MB
